# GSDM 1.1.0 on Google Colab

This notebook installs GSDM, reports the machine, runs the test suite and the benchmark, trains GVF on the
synthetic demo dataset with grouped validation, and shows the training dashboard.

**Read before choosing a runtime.** GSDM 1.1.0 runs on the CPU (pure NumPy). A GPU runtime such as an A100 is
detected by the hardware cell, but GSDM does not use it yet. A CPU runtime with high RAM gives the same results
for less. GPU support is planned for a later version.

## 1. Install

In [ ]:
REF = "main"  # or a release tag such as "v1.1.0" for a fixed version
!git clone --depth 1 --branch {REF} https://github.com/AnThophicous/gsdm.git
%cd gsdm
!pip install -q -e ".[dev]"

## 2. What machine is this?

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "no NVIDIA GPU"
!lscpu | grep -E "Model name|^CPU\(s\)|Flags" | cut -c1-200
!free -h | head -2
import numpy as np
np.show_runtime()

## 3. Tests (gradient checks, architecture rules, end-to-end)

In [ ]:
!python -m pytest -q

## 4. Benchmark

Compare these numbers with the reference low-end machine in the README.

In [ ]:
!python benchmarks/benchmark.py

## 5. How big can a model be?

`gvf capacity` computes parameters and memory without allocating anything. The first line is a realistic size
for this task; the second shows what 10 billion parameters would need.

In [ ]:
!gvf capacity --hidden-size 256 --reasoning-steps 4
!echo
!gvf capacity --target-parameters 1e10 --reasoning-steps 4

## 6. Train on the synthetic demo dataset

The demo dataset has one group per template, so validation only contains templates the model never saw.
Expect the diagnosis to say **memorizing**: a dozen templates are not enough to learn the patterns.
That is the detector working, not a bug.

In [ ]:
!python benchmarks/synthetic_dataset.py data --variants 6
!gvf train data/manifest.jsonl --output runs/gvf.npz --epochs 80 --hidden-size 64 --reasoning-steps 4     --validation-fraction 0.3 --patience 10

## 7. Dashboard

In [ ]:
import html
from IPython.display import HTML

page = open("runs/gvf.run/dashboard.html", encoding="utf-8").read()
HTML(f'<iframe srcdoc="{html.escape(page)}" style="width:100%;height:1100px;border:0"></iframe>')

## 8. Real data: CVEfixes census

[CVEfixes](https://zenodo.org/records/13118970) (v1.0.8, CC BY 4.0) collects real fixes of CVEs with the code
before and after each change. This section only **counts** how many usable Python examples exist for VC-1, VC-2
and VC-3 ([ADR-0010](https://github.com/AnThophicous/gsdm/blob/main/docs/adr/0010-cvefixes-dataset.md)). Nothing is
built or uploaded.

The release is a **12.7 GB** download and the SQLite database it becomes is larger, so everything goes to the Colab
disk, not to Google Drive. Check the free space first.

In [ ]:
!df -h /content | tail -1
!mkdir -p /content/cvefixes
!wget -c -q --show-progress -O /content/cvefixes/CVEfixes_v1.0.8.zip "https://zenodo.org/records/13118970/files/CVEfixes_v1.0.8.zip?download=1"

Import the dump into SQLite. The MD5 published by Zenodo is checked first; the dump is streamed out of the zip without extracting it.

In [ ]:
!python -m dataset_tools.cvefixes prepare --zip /content/cvefixes/CVEfixes_v1.0.8.zip --output /content/cvefixes/CVEfixes.db
!rm -f /content/cvefixes/CVEfixes_v1.0.8.zip  # frees 12.7 GB once the database exists

The census. It prints usable before/after pairs per category, how many repositories they come from, every discard
with its reason, graph sizes, and the most common other CWEs in Python. Download `census.json` and share it: those
numbers decide the next step.

In [ ]:
!python -m dataset_tools.cvefixes census /content/cvefixes/CVEfixes.db --report census.json
from google.colab import files
files.download("census.json")

## 9. Train on your own dataset

Put your files and a `manifest.jsonl` on Google Drive. Each line looks like
`{"path": "files/a.py", "labels": ["VC-1"], "group": "project-or-cve-id"}`.
Writing the output to Drive keeps the checkpoint if Colab disconnects; run the same command with `--resume`
to continue exactly where it stopped.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATASET = "/content/drive/MyDrive/gvf-dataset/manifest.jsonl"
OUTPUT = "/content/drive/MyDrive/gvf-runs/gvf.npz"
!mkdir -p /content/drive/MyDrive/gvf-runs
!gvf train "$DATASET" --output "$OUTPUT" --epochs 200 --hidden-size 128 --reasoning-steps 4 --patience 15

In [ ]:
# Continue an interrupted run (same manifest, same output). --epochs is the new total.
!gvf train "$DATASET" --output "$OUTPUT" --epochs 300 --resume

## 10. Download the results

In [ ]:
!cd runs && zip -qr ../gvf-demo-run.zip .
from google.colab import files
files.download("gvf-demo-run.zip")